# Targeted proteomics — PRM FASTA builder

Reads Skyline transition results, extracts UniProt accessions from protein names, fetches sequences from UniProt, and exports a FASTA file for spectral library building.

In [ ]:
import os
import pandas as pd
import numpy as np
from blood_proteome.fn import get_protein_sequences_batch, validate_fasta

## Load PRM transition results

In [ ]:
prm_results = pd.read_csv('targeted/Transition Results.csv')

prm_selected = (
    prm_results[['Peptide Modified Sequence', 'Protein Name']]
    .drop_duplicates()
)
prm_selected

## Unique proteins

In [ ]:
selected_proteins = prm_selected['Protein Name'].unique()
print(f"Unique proteins: {len(selected_proteins)}")

## Fetch sequences from UniProt and write FASTA

Accessions are parsed from the `sp|ACCESSION|ENTRY` format in the Protein Name column.

In [ ]:
def extract_accession(name):
    if pd.isna(name):
        return None
    parts = str(name).split('|')
    return parts[1].strip() if len(parts) >= 2 and parts[1].strip() else None


accessions = (
    prm_selected['Protein Name']
    .map(extract_accession)
    .dropna()
    .drop_duplicates()
    .tolist()
)
print(f"Unique accessions to fetch: {len(accessions)}")

entries = get_protein_sequences_batch(accessions, batch_size=50)
entry_by_acc = {
    e.get('primaryAccession', ''): e
    for e in entries
    if e.get('primaryAccession')
}
missing = [acc for acc in accessions if acc not in entry_by_acc]
print(f"Missing from UniProt: {len(missing)}")

In [ ]:
def build_header(entry):
    prot_name = 'Unknown protein'
    desc = entry.get('proteinDescription', {})
    if 'recommendedName' in desc and desc['recommendedName']:
        fn = desc['recommendedName'].get('fullName', '')
        prot_name = fn.get('value', prot_name) if isinstance(fn, dict) else fn or prot_name
    elif 'submittedName' in desc and desc['submittedName']:
        submitted = desc['submittedName']
        if isinstance(submitted, list) and submitted:
            fn = submitted[0].get('fullName', '')
            prot_name = fn.get('value', prot_name) if isinstance(fn, dict) else fn or prot_name

    accession = entry.get('primaryAccession', 'UNKNOWN')
    entry_name = str(entry.get('uniProtkbId') or entry.get('uniProtKBId') or f"{accession}_HUMAN")
    reviewed = entry.get('reviewed', None)
    is_reviewed = reviewed if isinstance(reviewed, bool) else 'Swiss-Prot' in entry.get('entryType', '')
    db_type = 'sp' if is_reviewed else 'tr'

    gene_name = ''
    genes = entry.get('genes', [])
    if isinstance(genes, list) and genes:
        gi = genes[0]
        if isinstance(gi, dict):
            gene_name = gi.get('geneName', {}).get('value', '') or gi.get('geneName', '')

    taxid = entry.get('organism', {}).get('taxonId') if isinstance(entry.get('organism'), dict) else None
    parts = [f"{db_type}|{accession}|{entry_name}", prot_name.strip(),
             'OS=Homo sapiens', f"OX={taxid or 9606}"]
    if gene_name:
        parts.append(f"GN={gene_name}")
    return ' '.join(parts)


out_dir = 'export'
os.makedirs(out_dir, exist_ok=True)
fasta_path = os.path.join(out_dir, 'prm_selected_uniprot.fasta')

with open(fasta_path, 'w') as fasta_file:
    for acc in accessions:
        entry = entry_by_acc.get(acc)
        if not entry:
            continue
        sequence = entry.get('sequence', {}).get('value', '')
        if not sequence:
            continue
        fasta_file.write(f">{build_header(entry)}\n")
        for i in range(0, len(sequence), 60):
            fasta_file.write(sequence[i:i + 60] + '\n')

print(f"FASTA written: {fasta_path}")
print(f"Proteins written: {sum(1 for acc in accessions if acc in entry_by_acc)}")

## Validate output

In [ ]:
validate_fasta(fasta_path)